In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [ ]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

columns = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome"
]

df = pd.read_csv(url, names=columns)

print(df.shape)
df.head()

In [ ]:
missing_cols = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

zero_counts = (df[missing_cols] == 0).sum()

print(zero_counts)

In [ ]:
df[missing_cols] = df[missing_cols].replace(0, np.nan)

In [ ]:
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [ ]:
imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns
)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
dt_raw = DecisionTreeClassifier(random_state=42)

dt_raw.fit(X_train, y_train)

train_pred = dt_raw.predict(X_train)
test_pred = dt_raw.predict(X_test)

print("Train Accuracy:", accuracy_score(y_train, train_pred))
print("Test Accuracy:", accuracy_score(y_test, test_pred))

print("Train F1:", f1_score(y_train, train_pred))
print("Test F1:", f1_score(y_test, test_pred))

In [ ]:
dt_scaled = DecisionTreeClassifier(random_state=42)

dt_scaled.fit(X_train_scaled, y_train)

scaled_pred = dt_scaled.predict(X_test_scaled)

print("Scaled Test Accuracy:",
      accuracy_score(y_test, scaled_pred))

print("Scaled Test F1:",
      f1_score(y_test, scaled_pred))

In [ ]:
raw_pred = dt_raw.predict(X_test)

print(
    "Same predictions:",
    np.array_equal(raw_pred, scaled_pred)
)

In [ ]:
dt_pruned = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=10,
    random_state=42
)

dt_pruned.fit(X_train, y_train)

pruned_pred = dt_pruned.predict(X_test)

print("Accuracy:",
      accuracy_score(y_test, pruned_pred))

print("F1:",
      f1_score(y_test, pruned_pred))

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

print("Accuracy:",
      accuracy_score(y_test, rf_pred))

print("F1:",
      f1_score(y_test, rf_pred))

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 4, 6, 8],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

rf_base = RandomForestClassifier(
    random_state=42
)

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("Best CV F1:",
      grid_search.best_score_)

In [ ]:
best_rf = grid_search.best_estimator_

tuned_pred = best_rf.predict(X_test)

print("Test Accuracy:",
      accuracy_score(y_test, tuned_pred))

print("Test F1:",
      f1_score(y_test, tuned_pred))

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Decision Tree - Unconstrained Raw",
        "Decision Tree - Unconstrained Scaled",
        "Decision Tree - Pre-Pruned",
        "Random Forest - Default",
        "Random Forest - Tuned"
    ],
    "Accuracy": [
        accuracy_score(y_test, raw_pred),
        accuracy_score(y_test, scaled_pred),
        accuracy_score(y_test, pruned_pred),
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, tuned_pred)
    ],
    "F1-score": [
        f1_score(y_test, raw_pred),
        f1_score(y_test, scaled_pred),
        f1_score(y_test, pruned_pred),
        f1_score(y_test, rf_pred),
        f1_score(y_test, tuned_pred)
    ]
})

results

In [ ]:
importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": best_rf.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

importance

In [ ]:
importance["Rank"] = range(1, len(importance) + 1)

importance